In [7]:
import json
import numpy as np
import ollama
import chromadb
from pathlib import Path

cleaned_documents_file = Path("../data/processed/cleaned_documents.json")
chunked_documents_file = Path("../data/processed/chunked_documents.json")
embedding_file = Path("../data/processed/embeddings.npy")
chroma_path = Path("../data/chroma")

print("Setup complete")

Setup complete


In [ ]:
#loading documents

with open(cleaned_documents_file, "r", encoding="utf-8") as f:
    cleaned_documents = json.load(f)

print("Total documents:", len(cleaned_documents))

Total documents: 1109


In [13]:
#loading chunks 

with open(chunked_documents_file, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

print("Total chunks:", len(all_chunks))

Total chunks: 2798


In [14]:

#loading embeddings 

embeddings_array = np.load(embedding_file)

print("Embeddings shape:", embeddings_array.shape)

Embeddings shape: (2798, 768)


In [15]:

print("Chunks:", len(all_chunks))
print("Embeddings:", len(embeddings_array))

assert len(all_chunks) == len(embeddings_array)

print("Chunk and embedding counts match")

Chunks: 2798
Embeddings: 2798
Chunk and embedding counts match


In [16]:
client = chromadb.PersistentClient(
    path=str(chroma_path)
)

print("ChromaDB client created successfully")

ChromaDB client created successfully


In [17]:
collection = client.get_or_create_collection(
    name="documents"
)

print("Collection:", collection.name)
print("Current count:", collection.count())

Collection: documents
Current count: 0


In [18]:
batch_size = 100

for start in range(0, len(all_chunks), batch_size):

    end = min(start + batch_size, len(all_chunks))

    batch_chunks = all_chunks[start:end]
    batch_embeddings = embeddings_array[start:end]

    collection.add(
        ids=[chunk["chunk_id"] for chunk in batch_chunks],
        embeddings=batch_embeddings.tolist(),
        documents=[chunk["text"] for chunk in batch_chunks],
        metadatas=[
            {
                "doc_id": chunk["doc_id"],
                "source": chunk["source"],
                "chunk_index": chunk["chunk_index"]
            }
            for chunk in batch_chunks
        ]
    )

    print(f"Inserted {end}/{len(all_chunks)} chunks")

Inserted 100/2798 chunks
Inserted 200/2798 chunks
Inserted 300/2798 chunks
Inserted 400/2798 chunks
Inserted 500/2798 chunks
Inserted 600/2798 chunks
Inserted 700/2798 chunks
Inserted 800/2798 chunks
Inserted 900/2798 chunks
Inserted 1000/2798 chunks
Inserted 1100/2798 chunks
Inserted 1200/2798 chunks
Inserted 1300/2798 chunks
Inserted 1400/2798 chunks
Inserted 1500/2798 chunks
Inserted 1600/2798 chunks
Inserted 1700/2798 chunks
Inserted 1800/2798 chunks
Inserted 1900/2798 chunks
Inserted 2000/2798 chunks
Inserted 2100/2798 chunks
Inserted 2200/2798 chunks
Inserted 2300/2798 chunks
Inserted 2400/2798 chunks
Inserted 2500/2798 chunks
Inserted 2600/2798 chunks
Inserted 2700/2798 chunks
Inserted 2798/2798 chunks


In [19]:
print("Chroma collection count:", collection.count())

Chroma collection count: 2798


In [20]:
result = collection.get(
    ids=[all_chunks[0]["chunk_id"]],
    include=["embeddings", "documents", "metadatas"]
)

print("ID:", result["ids"][0])
print("Embedding dimensions:", len(result["embeddings"][0]))
print("Document:", result["documents"][0][:300])
print("Metadata:", result["metadatas"][0])

ID: NCT06909825_0
Embedding dimensions: 768
Document: Title: FPI-2265 (225Ac-PSMA-I&T) and Olaparib for Patients With Metastatic Castration-Resistant Prostate Cancer (mCRPC) Official Title: A Phase 2, Open-label, Multi-centre Study of FPI-2265 (225Ac-PSMA-I&T) and Olaparib in Participants With Metastatic Castration Resistant Prostate Cancer (mCRPC) Con
Metadata: {'source': 'clinical_trials', 'doc_id': 'NCT06909825', 'chunk_index': 0}
